# Mr.X BC GNN validation

Notebook Kaggle per validare il checkpoint `mrx_rgnn_bc_best_*.pt`.

Confronta:
- Mr.X random vs detective heuristic;
- Mr.X GNN BC vs detective heuristic;
- Mr.X GNN BC vs detective GNN frozen;
- Mr.X MCTS vs detective GNN frozen.

Output: JSON in `/kaggle/working/mrx_bc_validation.json`.

## 1. Setup repo

In [ ]:
%%bash
set -e
cd /kaggle/working
python -m pip install -q networkx numpy pandas tqdm scipy torch || true

REPO_DIR=/kaggle/working/scotland_yard
if [ ! -d "$REPO_DIR" ]; then
  CANDIDATE=$(find /kaggle/input -maxdepth 4 -type d -name scotland_yard | head -n 1 || true)
  if [ -n "$CANDIDATE" ]; then
    echo "Copying repo from Kaggle input: $CANDIDATE"
    cp -R "$CANDIDATE" "$REPO_DIR"
  else
    echo "Cloning repo from GitHub"
    git clone --depth 1 https://github.com/Jacopo888/scotland_yard.git "$REPO_DIR"
  fi
fi
ls "$REPO_DIR"


: 

## 2. Locate checkpoints

In [ ]:
import glob
import os
from pathlib import Path

REPO_DIR = Path('/kaggle/working/scotland_yard')
os.chdir(REPO_DIR)

mrx_candidates = []
mrx_candidates += sorted(glob.glob('/kaggle/input/**/mrx_rgnn_bc_best_*.pt', recursive=True))
mrx_candidates += sorted(glob.glob(str(REPO_DIR / 'Notebook' / 'mrx_rgnn_bc_best_*.pt')))

det_candidates = []
det_candidates += sorted(glob.glob('/kaggle/input/**/rgnn_ppo_best_*.pt', recursive=True))
det_candidates += sorted(glob.glob(str(REPO_DIR / 'Notebook' / 'rgnn_ppo_best_*.pt')))

assert mrx_candidates, 'Missing Mr.X checkpoint mrx_rgnn_bc_best_*.pt'
assert det_candidates, 'Missing detective checkpoint rgnn_ppo_best_*.pt'

MRX_CKPT = sorted(set(mrx_candidates), key=lambda p: os.path.basename(p))[-1]
DET_CKPT = sorted(set(det_candidates), key=lambda p: os.path.basename(p))[-1]
print('Mr.X checkpoint:', MRX_CKPT)
print('Detective checkpoint:', DET_CKPT)


## 3. Run validation

In [ ]:
import subprocess
import sys
import torch

OUTPUT = '/kaggle/working/mrx_bc_validation.json'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Validation device:', DEVICE)

cmd = [
    sys.executable, 'validate_gnn_mrx.py',
    '--mrx-checkpoint', MRX_CKPT,
    '--detective-checkpoint', DET_CKPT,
    '--device', DEVICE,
    '--quiet',
    '--output', OUTPUT,
    '--suite', 'heuristic_vs_random:heuristic:random:100',
    '--suite', 'heuristic_vs_mrx_gnn_bc:heuristic:gnn:200',
    '--suite', 'gnn_detectives_vs_mrx_gnn_bc:gnn:gnn:200',
    '--suite', 'gnn_detectives_vs_mcts_fixed:gnn:mcts:100:15:25',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('Saved:', OUTPUT)


## 4. Inspect results

In [ ]:
import json
import pandas as pd

with open('/kaggle/working/mrx_bc_validation.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

rows = []
for suite in data['suites'].values():
    rows.append({
        'suite': suite['name'],
        'detectives': suite['detectives'],
        'mrx': suite['mrx'],
        'games': suite['n_games'],
        'mrx_winrate': suite['mrx_winrate'],
        'avg_final_turn': suite['avg_final_turn'],
        'illegal_actions': suite['illegal_actions'],
        'tickets': suite['ticket_counts'],
    })
df = pd.DataFrame(rows)
display(df)
print(json.dumps(data['mrx_checkpoint_metadata'], indent=2))
